## Context

This notebook accompanies User Guide section 27 (Testing your agent).  It exists to demonstrate, end to end, that the build-side surface this guide covers and the test-side surface in `knowlytix/harness/testing/` are *separate* pipelines that share only the underlying GMS primitives.  One run from ingest through release gate, against a deterministic mock evaluator, then a clear handoff to the deeper testing material.  No API key required.

In [ ]:
# Bootstrap: portable repo-root locator — walks up from CWD looking for
# tutorials/user_guide/notebooks/_shared.py. Works in both the source
# clone (knowlytix/GMS) and the public mirror clone (knowlytix/gmsh).
import os
import sys
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'tutorials' / 'user_guide' / 'notebooks' / '_shared.py').exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from tutorials.user_guide.notebooks import _shared as ug
ug.bootstrap_runtime()  # stubs modeva if absent
import torch
DEVICE = ug.DEVICE
print(f'Device: {DEVICE}, CUDA available: {torch.cuda.is_available()}')


### Concept: build-side ends at deployment, test-side starts there

Sections 1 through 26 of this guide build a governed agent: ingest knowledge, train geometry, calibrate thresholds, sign a `GovernanceBundle`, freeze it.  At that point the agent is ready to deploy.  Testing is a different surface: `knowlytix/harness/testing/`, the GMSH testing harness.  GMSH ingests the same markdown, builds the same GMS store, but then *generates test questions from it* via DOE factor sweeps and grades agent answers using the very same primitives — `score_triple`, `tension_energy`, `check_holonomy`, `lookup_enm` — that the agent's claim verifier uses in production.

The handoff between the two surfaces is the GMS store on disk.  In this notebook we point GMSH at the prebuilt regex-mode `ds_practices_store` fixture from Phase 0.  When `bench.ingest()` runs it discovers the existing store and loads it — no LLM is invoked, no API key is required.  Then `bench.generate_questions()` walks the store with the eight default geometric generators (plausibility, tension, holonomy, link prediction, multi-hop transport, phase comparison, conditional, procedural) plus five taxonomy generators (boundary, adversarial, robustness, recovery, governance).  `bench.run(evaluator=...)` calls the evaluator on every question, decomposes each answer into typed claims, and routes the claims to dedicated verifiers.  `bench.analyze(result)` then runs DOE factor analysis over the per-question outcomes so a regression can be attributed to a factor rather than to noise.

The final HTML dashboard returned by `bench.generate_report(...)` is structured rather than narrative.  An auditor can inspect *why* the gate passed or failed — which threshold drove the verdict, which factor's main effect was significant — without rerunning the benchmark.

### One end-to-end run, no API key

We use the prebuilt fixture as the GMS store and `create_mock_evaluator()` from `demo/mock_agent.py` as the system under test.  The mock evaluator is deterministic, has tool-use logic, and returns answers from canned tool outputs.  Swap it for any `(question, context) -> answer` callable to test a real agent.

In [ ]:
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig
from knowlytix.harness.fixtures.mock_agent import create_mock_evaluator

FIXTURE_PARENT = str(ug.FIXTURES_DIR)
KNOWLEDGE = str(
    ug.REPO_ROOT / 'knowlytix' / 'harness' / 'governance' / 'knowledge'
    / 'data_science_practices.md')

config = DOEHarnessConfig(
    markdown_path=KNOWLEDGE,
    stores_dir=FIXTURE_PARENT,
    store_name='ds_practices_store',
    factor_group='quick_screen',
    n_runs=4,
    max_per_category=2,
    seed=42,
    target_type='agent',
    target_model='mock-agent-v1',
    enable_tracing=False,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
with ug.quiet():
    store = bench.ingest()  # loads existing fixture; no LLM
stats = store.stats()
print(f"Store loaded: {stats['entities']} entities, "
      f"{stats['triples']} triples, "
      f"{stats['enm_entries']} ENM entries")

In [ ]:
with ug.quiet():
    questions = bench.generate_questions()
print(f'DOE generated {len(questions)} expanded questions')

evaluator = create_mock_evaluator(recovery_mode='fallback')
with ug.quiet():
    result = bench.run(evaluator=evaluator)
print(
    f'accuracy = {result.accuracy:.1%} '
    f'({result.total_correct}/{result.total_questions})'
)

In [ ]:
analyzer = bench.analyze(result)
report_path = str(
    ug.NOTEBOOKS_DIR / '_figures' / '27_doe_report.html')
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with ug.quiet():
    bench.generate_report(
        result, analyzer=analyzer, output_path=report_path)
print(f'HTML dashboard written to {report_path}')

### Visualization: per-category accuracy

The factor-effect bar chart below summarizes which question categories the mock evaluator handled well versus poorly.  Strong horizontal variation across categories is normal; it tells you which generator surface stresses the system under test.  In a real run you would invoke `analyzer.run_logistic()` on top of this to extract factor main effects.  Here we plot per-category accuracy because the mock has only one factor of interest (the recovery mode) which is held fixed.

In [ ]:
import matplotlib.pyplot as plt
df = result.to_dataframe()
if df.empty:
    raise RuntimeError('No DOE result rows; check evaluator')
by_cat = df.groupby('category')['correct'].agg(['sum', 'count'])
by_cat['rate'] = by_cat['sum'] / by_cat['count']
by_cat = by_cat.sort_values('rate')

fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(by_cat))))
ax.barh(by_cat.index, by_cat['rate'], color='tab:blue')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1,
           label='0.5 baseline')
ax.set_xlim(0.0, 1.0)
ax.set_xlabel('accuracy')
ax.set_title(
    f'NB 27 — DOE per-category accuracy '
    f'(overall {result.accuracy:.1%})'
)
ax.legend(loc='lower right')
plt.tight_layout()
fig_path = str(
    ug.NOTEBOOKS_DIR / '_figures' / '27_factor_effects.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
fig.savefig(fig_path, dpi=120)
plt.show()

### Exercises

1. Replace `create_mock_evaluator()` with `make_evaluator('llm', 'claude-opus-4-7')` (requires `ANTHROPIC_API_KEY`) and rerun.  How does the accuracy curve shift across categories?
2. Set `factor_group='full_factorial'` and `n_runs=16`, then call `analyzer.run_logistic()`.  Which factor's main effect crosses the 95% confidence boundary?
3. Add `enable_severity_classification=True` to the config and inspect `result.questions[i].severity` after the run.  Which severity tier do hallucinated answers land in?

### What you should now be able to do

- Drive a one-line end-to-end DOE benchmark against any callable evaluator using a prebuilt GMS fixture
- Distinguish the build-side surface (`knowlytix/harness/governance/`, `gms/`, `knowlytix/knowledge/`) from the test-side surface (`knowlytix/harness/testing/`)
- Read a per-category accuracy chart and recognize which generator surfaces the agent struggles on
- Locate the deeper testing material when an audit, regression, or release gate decision needs more depth

### Where to next?

This notebook is a *pointer*, not a substitute.  GMSH has 38 sections of tutorial material in [`knowlytix/harness/testing/USER_GUIDE.md`](../../knowlytix/harness/testing/USER_GUIDE.md) and 19 chapters of monograph material in [`gmsh_testing_ext/`](../../gmsh_testing_ext/).  When your role includes signing release gates or attesting to GMSTO numbers, read the testing monograph end-to-end.

**Cross-references:**

- Build-side telemetry: USER_GUIDE.md §22 Telemetry & Tracing; `gms_agent_builder_ext/chapters/ch14.tex`
- Build-side release gates: USER_GUIDE.md §23 Release Gates & SPC; `gms_agent_builder_ext/chapters/ch15.tex`
- Test-side benchmark API: `knowlytix/harness/testing/USER_GUIDE.md` §4; `gmsh_testing_ext/chapters/ch04.tex`
- Test-side end-to-end walkthrough: `knowlytix/harness/testing/USER_GUIDE.md` §34

In [ ]:
# Final assertion: the harness produced at least one
# successful evaluation against the mock agent.
assert result.total_questions >= 1, (
    f'NB 27: expected >= 1 question, got '
    f'{result.total_correct}/{result.total_questions}'
)
# TODO(p7-execute): tighten threshold once a real run is observed.
print(f'NB 27 PASS: '
      f'{result.total_correct}/{result.total_questions} '
      f'correct ({result.accuracy:.1%})')